# DroneRoute · Y-2 · Colab'da ilk ince ayar

**Ne yapıyoruz?** Hazır YOLO modelini kendi pilot setimizle (UAVid, 270 resim) biraz daha eğitiyoruz,
özellikle **bina ve ağacı** yukarıdan daha iyi bulsun diye. Sonra eski model (YOLOE, sözlüklü) ile yeni
modeli **aynı test resimlerinde** karşılaştırıyoruz.

**Başlamadan önce (bir kere):**
1. Üst menü **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU**.
2. Ekibin Drive klasörü (`DroneRoute_veri`) sana "Benimle paylaşılanlar"da görünüyorsa Colab onu göremez:
   Drive'da klasöre sağ tık → **Düzenle → Kısayol ekle → Drive'ım**. Klasörün adını DEĞİŞTİRME (ekibin ortak klasörü).
3. Drive'da şunlar olmalı:
   - `DroneRoute_veri/veri/uavid_yolo.zip`  (bilgisayarındaki `veri/uavid` + `veri/bolme.json`, zip'li)
   - `DroneRoute_veri/modeller/droneroute_yoloe.pt`  (bilgisayarındaki `models/droneroute_yoloe.pt`, karşılaştırma için)

Sonra hücreleri yukarıdan aşağı sırayla çalıştır (her hücrenin solundaki ▶).
Eğitim T4'te yaklaşık 30-45 dakika sürer. Bağlantı koparsa "Kaldığı yerden devam" hücresine bak.

In [ ]:
# 1) GPU var mı, Drive'ı bağla
import os, torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "[!] YOK: Çalışma zamanı türünden T4 GPU seç")
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 2) Ayarlar: yollar ve eğitim seçenekleri (gerekirse sadece burayı değiştir)
DRIVE = "/content/drive/MyDrive/DroneRoute_veri"     # ekip klasörünün Drive'ım'daki kısayolu
VERI_ZIP = f"{DRIVE}/veri/uavid_yolo.zip"              # maske_cevir çıktısı (zip)
ESKI_MODEL = f"{DRIVE}/modeller/droneroute_yoloe.pt"   # karşılaştırma: sözlüklü hazır model
CIKIS = f"{DRIVE}/modeller/y2_egitim"                  # eğitim sonuçları Drive'a (bağlantı koparsa kaybolmasın)
YEREL = "/content/veri"                                # veri Colab diskine açılır (Drive'dan okumak çok yavaş)

TEMEL_MODEL = "yolo11n-seg.pt"   # küçük, segmentasyonlu hazır model (COCO'da eğitilmiş)
EPOCH = 50                       # bütün eğitim seti kaç kez gösterilecek
IMGSZ = 1280                     # resim boyu: 4K resimlerde insanlar küçük, 640 çok az kalır
BATCH = 8                        # T4'te 1280 için güvenli
eksik = [yol for yol in (VERI_ZIP, ESKI_MODEL) if not os.path.exists(yol)]
for yol in (VERI_ZIP, ESKI_MODEL):
    print(("✓ " if yol not in eksik else "[!] YOK: ") + yol)
if eksik:   # neyin göründüğünü göster: klasör adı mı farklı, kısayol mu yok?
    print("\nDrive'ım'da görünenler:", sorted(os.listdir("/content/drive/MyDrive")))
    if os.path.isdir(DRIVE):
        print(f"{DRIVE} içinde:", sorted(os.listdir(DRIVE)))
    print("→ Ekip klasörünün kısayolu Drive'ım'da mı, adı DRIVE satırındakiyle aynı mı? Gerekirse DRIVE'ı düzelt, bu hücreyi tekrar çalıştır.")

In [ ]:
# 3) Kütüphane + proje kodu (sınıf adları ve eşleme drone_yolo.py'den gelsin, iki yerde ayrı yazılmasın)
!pip install -q ultralytics
!rm -rf /content/droneroute && git clone -q --depth 1 https://github.com/Rakcery/droneroute /content/droneroute
import sys
sys.path.insert(0, "/content/droneroute")
import drone_yolo
SINIFLAR = list(drone_yolo.CLASSES)            # ['vinc', 'bina', 'agac', 'arac', 'insan', 'kus'] (S-5 sırası)
print(SINIFLAR)

In [ ]:
# 4) Veriyi aç ve YOLO'ya tarif et (data.yaml)
#    Klasörü yeniden düzenlemiyoruz: bolme.json'daki her sahnenin resimlerini listelere yazıyoruz.
#    YOLO etiketi resmin yolundaki "images" -> "labels" değişimiyle buluyor (seq1/images/x.jpg -> seq1/labels/x.txt).
import json, glob, zipfile, shutil

def veriyi_hazirla(zip_yolu, hedef, siniflar):
    """Zip'i açar, bolme.json'a göre egitim/dogrulama/test listelerini ve data.yaml'ı yazar."""
    isaret = os.path.join(hedef, ".acildi")      # zip tam açıldıysa bu dosya yazılır
    if not os.path.exists(isaret):               # hiç açılmamış ya da yarıda kalmış: temizle, baştan aç
        shutil.rmtree(hedef, ignore_errors=True)
        print("zip açılıyor (1-2 dk)...")
        with zipfile.ZipFile(zip_yolu) as z:
            z.extractall(hedef)
        open(isaret, "w").close()
    # Zip'te araya bir klasör girmiş olabilir: bolme.json nerede ise kök orası
    kok = os.path.dirname(glob.glob(os.path.join(hedef, "**", "bolme.json"), recursive=True)[0])
    with open(os.path.join(kok, "bolme.json"), encoding="utf-8") as f:
        bolme = json.load(f)
    sayilar = {}
    for b in ("egitim", "dogrulama", "test"):
        resimler = []
        for sahne in bolme[b]:
            resimler += sorted(glob.glob(os.path.join(kok, sahne, "images", "*.jpg")))
        with open(os.path.join(kok, f"{b}.txt"), "w") as f:
            f.write("\n".join(resimler) + "\n")
        sayilar[b] = len(resimler)
    yaml_yolu = os.path.join(kok, "data.yaml")
    with open(yaml_yolu, "w", encoding="utf-8") as f:
        f.write(f"path: {kok}\ntrain: egitim.txt\nval: dogrulama.txt\ntest: test.txt\nnames:\n")
        f.writelines(f"  {i}: {ad}\n" for i, ad in enumerate(siniflar))
    print("resim sayısı:", sayilar)
    return yaml_yolu, kok

DATA_YAML, VERI_KOK = veriyi_hazirla(VERI_ZIP, YEREL, SINIFLAR)
print(open(DATA_YAML).read())

In [ ]:
# 5) EĞİTİM (T4'te ~30-45 dk). Sonuçlar Drive'da: modeller/y2_egitim/ilk/
from ultralytics import YOLO
assert os.path.exists(DATA_YAML), "data.yaml yok: önce 4. hücreyi çalıştır"
model = YOLO(TEMEL_MODEL)
model.train(data=DATA_YAML, epochs=EPOCH, imgsz=IMGSZ, batch=BATCH,
            project=CIKIS, name="ilk", exist_ok=True,
            patience=15,            # doğrulama 15 tur iyileşmezse erken dur (aşırı öğrenmeye karşı)
            seed=0, plots=True)

**Bağlantı koptuysa:** 1-4. hücreleri tekrar çalıştır, sonra 5. hücre yerine aşağıdakini çalıştır
(başındaki `#` işaretlerini silerek). Eğitim son kaydedildiği turdan devam eder.

In [ ]:
# 5b) Kaldığı yerden devam (sadece bağlantı koptuysa)
# from ultralytics import YOLO
# YOLO(f"{CIKIS}/ilk/weights/last.pt").train(resume=True)

In [ ]:
# 6) Karşılaştırma: eski model (YOLOE, sözlüklü) vs ince ayarlı model, TEST setinde (bir kez!)
#    Ölçü alan bazında: her sınıf için "gerçekte o sınıf olan alanın ne kadarını buldu" (duyarlılık) ve
#    "o sınıf dediği alanın ne kadarı gerçekten o sınıf" (kesinlik). Kutu kutu değil, çünkü UAVid bitişik
#    binaları tek bölge etiketliyor; planlayıcı da zaten alana (hücreye) bakıyor. Y-0'daki hücre ölçümüne benzer.
import cv2, numpy as np

OLCEK = 8   # 4K maskeyi 1/8'e küçültüp karşılaştır (480x270): hız için, sonuç pek değişmez

def etiket_maskesi(etiket_yolu, W, H):
    """YOLO segmentasyon etiketini sınıf başına maskeye çevirir (küçültülmüş)."""
    w, h = W // OLCEK, H // OLCEK
    m = {s: np.zeros((h, w), np.uint8) for s in range(len(SINIFLAR))}
    for satir in open(etiket_yolu, encoding="utf-8").read().split("\n"):
        p = satir.split()
        if len(p) < 7:
            continue
        pts = (np.array(p[1:], float).reshape(-1, 2) * [w, h]).round().astype(np.int32)
        cv2.fillPoly(m[int(p[0])], [pts], 1)
    return m

def tahmin_maskesi(model, resim_yolu, W, H, esle, imgsz):
    """Modelin tahminlerini (segmentasyon) bizim sınıflara eşleyip sınıf başına maskeye çevirir."""
    w, h = W // OLCEK, H // OLCEK
    m = {s: np.zeros((h, w), np.uint8) for s in range(len(SINIFLAR))}
    r = model.predict(resim_yolu, imgsz=imgsz, conf=0.25, verbose=False)[0]
    if r.masks is None:
        return m
    for poli, k in zip(r.masks.xy, r.boxes.cls.tolist()):
        s = esle.get(int(k))
        if s is None or len(poli) < 3:
            continue
        cv2.fillPoly(m[s], [(np.array(poli) / OLCEK).round().astype(np.int32)], 1)
    return m

def karsilastir(model, esle, imgsz, liste_yolu):
    """Test listesindeki her resimde alan bazında TP/FP/FN toplar; sınıf başına kesinlik/duyarlılık döner."""
    T = {s: [0, 0, 0] for s in range(len(SINIFLAR))}
    for resim in open(liste_yolu).read().split():
        H, W = cv2.imread(resim).shape[:2]
        etiket = resim.replace(f"{os.sep}images{os.sep}", f"{os.sep}labels{os.sep}").rsplit(".", 1)[0] + ".txt"
        g, t = etiket_maskesi(etiket, W, H), tahmin_maskesi(model, resim, W, H, esle, imgsz)
        for s in T:
            gs, ts = g[s].astype(bool), t[s].astype(bool)
            T[s][0] += int((gs & ts).sum())     # gerçekte var, model de buldu
            T[s][1] += int((~gs & ts).sum())    # gerçekte yok, model "var" dedi (yanlış alarm)
            T[s][2] += int((gs & ~ts).sum())    # gerçekte var, model kaçırdı
    out = {}
    for s, (tp, fp, fn) in T.items():
        if tp + fn > 0:                      # test setinde bu sınıf hiç yoksa yazma (vinç, kuş)
            out[SINIFLAR[s]] = (tp / max(tp + fp, 1), tp / max(tp + fn, 1))
    return out

TEST = os.path.join(VERI_KOK, "test.txt")
sonuc = {}
# Eski model: sözlüklü YOLOE. Onun sınıf adları "building", "tree canopy"... drone_yolo.NAME2CLASS bunları bizimkine çevirir.
if os.path.exists(ESKI_MODEL):
    eski = YOLO(ESKI_MODEL)
    esle = {int(i): SINIFLAR.index(drone_yolo.NAME2CLASS[n.lower()]) for i, n in eski.names.items()
            if n.lower() in drone_yolo.NAME2CLASS}
    sonuc["hazir"] = karsilastir(eski, esle, IMGSZ, TEST)    # adil olsun: iki model de aynı resim boyunda
else:
    print("[!] Eski model Drive'da yok, sadece yeni model ölçülecek")
yeni = YOLO(f"{CIKIS}/ilk/weights/best.pt")
sonuc["ince_ayar"] = karsilastir(yeni, {i: i for i in range(len(SINIFLAR))}, IMGSZ, TEST)

print(f"{'sınıf':6s} | {'model':9s} | kesinlik | duyarlılık")
for ad in SINIFLAR:
    for yontem, d in sonuc.items():
        if ad in d:
            print(f"{ad:6s} | {yontem:9s} | {d[ad][0]:.3f}    | {d[ad][1]:.3f}")

In [ ]:
# 7) Sonuçları kaydet: CSV (Sözleşme S-6) + modeli Drive'a kopyala
import csv
csv_yolu = f"{DRIVE}/modeller/y-2_ince_ayar.csv"
SUTUNLAR = ["kart", "deney", "yontem", "sahne", "metrik", "deger", "birim"]
with open(csv_yolu, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=SUTUNLAR)
    w.writeheader()
    for yontem, d in sonuc.items():
        for ad, (k, r) in d.items():
            w.writerow(dict(kart="Y-2", deney="ince_ayar_alan", yontem=yontem, sahne=ad, metrik="kesinlik", deger=round(k, 4), birim="oran"))
            w.writerow(dict(kart="Y-2", deney="ince_ayar_alan", yontem=yontem, sahne=ad, metrik="duyarlilik", deger=round(r, 4), birim="oran"))
    # YOLO'nun kendi ölçüsü de dursun (doğrulama seti, kutu ve maske mAP50)
    m = yeni.val(data=DATA_YAML, imgsz=IMGSZ, split="val", verbose=False)
    w.writerow(dict(kart="Y-2", deney="yolo_val", yontem="ince_ayar", sahne="hepsi", metrik="kutu_mAP50", deger=round(float(m.box.map50), 4), birim="oran"))
    w.writerow(dict(kart="Y-2", deney="yolo_val", yontem="ince_ayar", sahne="hepsi", metrik="maske_mAP50", deger=round(float(m.seg.map50), 4), birim="oran"))
shutil.copy(f"{CIKIS}/ilk/weights/best.pt", f"{DRIVE}/modeller/kendi_y2.pt")
print("CSV  :", csv_yolu)
print("Model:", f"{DRIVE}/modeller/kendi_y2.pt")
print(open(csv_yolu).read())

**Bitti.** Şimdi bilgisayarında:
1. Drive'dan `modeller/kendi_y2.pt` → proje klasöründe `models/kendi_y2.pt` (depoya girmez, `.gitignore`'da).
2. Drive'dan `modeller/y-2_ince_ayar.csv` → `sonuclar/y-2_ince_ayar.csv`.
3. `python drone_yolo.py --resim ornekler/sokak_otobus.jpg --model models/kendi_y2.pt` ile dene.
4. `python tests/test_yolo.py` → TAMAM kalsın.

Ayrıca `modeller/y2_egitim/ilk/` içindeki `results.png` (eğitim eğrileri) ve `confusion_matrix.png`'e bak:
eğitim kaybı düşerken doğrulama kaybı yükseliyorsa model aşırı öğreniyor demektir.